# 02 — A credit portfolio, not an equity portfolio with bond labels

This is where the project stops being a re-run of the equity blueprint.
Three things change, and each one changes the answer:

1. **The instruments have fixed-income mechanics.** Every bond is priced from
   its own semiannual cash-flow schedule; duration and convexity fall out of
   that schedule. Spread duration, DTS and benchmark duration contributions
   are then arithmetic, not assumptions.
2. **The risk model is credit-shaped.** Spreads move *proportionally* to their
   level (the empirical fact behind DTS), the widening tail is fatter than the
   tightening one, rates rally when credit gaps, and **issuers default** —
   jointly, driven by the same systematic factor that widens spreads.
3. **The constraints are a credit desk's limit sheet**: rating bands, sector
   caps, per-issuer concentration, duration matching against the index (by
   maturity cell *and* in aggregate), a two-sided DTS band, and a turnover
   budget.

All of it stays linear, so the Rockafellar–Uryasev objective is still an LP.

In [1]:
import sys
import warnings
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO))

# CVXPY's bound propagation emits a harmless RuntimeWarning on the
# intentionally free VaR variable; silenced so stored notebook output
# carries no local filesystem paths.
warnings.filterwarnings("ignore", category=RuntimeWarning, module="cvxpy")

import numpy as np
import pandas as pd

from src.check import empirical_cvar
from src.credit import (credit_constraint_builder, default_limits,
                        exposure_report, portfolio_stats, worst_violation)
from src.model import build_mean_cvar
from src.scenarios import credit_scenarios, warmup_gpu
from src.universe import make_universe, summarize

pd.set_option("display.width", 120)
N_BONDS, N_SCEN, ALPHA = 500, 20_000, 0.95

## 1. The universe

Seeded and fully synthetic — reproducible with no data licence. The
calibration knobs (spread level by rating, sector multipliers, curve, issue
sizes) live in `CalibrationConfig`, which documents the public series each
field would be fitted to if a user chooses to.

In [2]:
uni = make_universe(n_bonds=N_BONDS, seed=7)
print(f"{uni.n_bonds} bonds, {uni.bonds['issuer_id'].nunique()} issuers, "
      f"{uni.bonds['sector'].nunique()} sectors")
print(f"incumbent portfolio is {uni.one_way_distance_to_benchmark():.1%} "
      f"one-way from the index")
uni.bonds[["bond_id", "issuer_id", "sector", "rating", "maturity", "coupon",
           "ytm", "oas_bp", "price", "mod_duration", "convexity"]].head(8)

500 bonds, 200 issuers, 8 sectors
incumbent portfolio is 24.9% one-way from the index


,bond_id,issuer_id,sector,rating,maturity,coupon,ytm,oas_bp,price,mod_duration,convexity
0,B00000,ISS0000,Healthcare,BBB,30.0,0.07500,0.061961,179.611516,117.670008,13.106096,270.398731
1,B00001,ISS0000,Healthcare,BBB,20.5,0.07250,0.057951,151.380652,117.322937,11.374414,183.888539
2,B00002,ISS0001,Consumer,BB,9.5,0.06000,0.069467,281.673094,93.495222,7.046809,61.862986
3,B00003,ISS0001,Consumer,BB,3.0,0.07500,0.067620,271.204451,101.973825,2.654806,8.715209
4,B00004,ISS0001,Consumer,BB,30.0,0.07125,0.068260,242.598355,103.795740,12.602156,255.093528
5,B00005,ISS0002,Energy,BBB,3.0,0.05750,0.054226,137.263069,100.895258,2.725220,9.068246
6,B00006,ISS0003,Healthcare,A,20.0,0.06250,0.055214,124.635668,108.756738,11.713925,190.363564
7,B00007,ISS0003,Healthcare,A,20.0,0.05000,0.054937,121.865626,94.053736,12.285156,204.931292


In [3]:
summarize(uni)   # the index: market-value weighted, as a real benchmark is

,weight,oas_bp,duration,dts,n_bonds
rating,,,,,
AAA,0.019112,58.996986,9.270942,10.572109,9.0
AA,0.067573,62.735348,6.470899,28.660457,38.0
A,0.297909,98.610227,6.342918,191.542422,147.0
BBB,0.444629,145.516674,6.533270,443.356424,221.0
BB,0.123798,261.800037,5.033140,172.521069,62.0
B,0.046978,362.093724,4.332465,74.671749,23.0


## 2. Excess return, not total return — the choice that decides everything

A 6-duration investment-grade book carries about **twice** as much monthly
risk from rates as from spreads:

$$D \times \sigma_{\text{rates}} = 6.24 \times 25\text{bp} = 1.56\%
\qquad\text{vs}\qquad
D \times s \times \sigma_{\text{rel}} = 6.24 \times 149\text{bp} \times 9.4\% = 0.87\%$$

The first version of this notebook optimized **total** return, and an
adversarial review of the model measured the consequence: **69% of the
apparent CVaR improvement was coming from shortening duration**, with the
credit tilt — the entire point of the project — nearly free of consequence. A
34% cut in DTS bought 3% of CVaR. It was a duration-timing machine wearing a
credit costume.

The fix is what credit desks actually do: optimize **excess return over
duration-matched Treasuries**, so rates are matched to the index and the
objective is purely credit. For a fixed-rate bullet the Treasury leg cancels
exactly:

$$r_{\text{excess}} = s\,h \;-\; SD \cdot \Delta s \;+\; \tfrac12 C (\Delta s)^2 \quad (+\ \text{default jump})$$

In [4]:
warmup_gpu(uni)   # real code path: context + this module's kernels
R, mu, info = credit_scenarios(uni, n_scen=N_SCEN, seed=11, use_gpu=True)
print(f"{info['device']}: {info['seconds']:.3f}s  ({info['return_mode']} returns)")
print(f"default rate/bond {info['realized_default_rate_per_bond']:.4%} "
      f"(target {info['target_default_rate_per_bond']:.4%}, threshold "
      f"calibrated on an independent draw)")
print(f"{info['scenarios_with_any_default']:.1%} of scenarios contain "
      f"at least one default")
print(f"mu vs realized scenario mean: max gap "
      f"{info['mu_vs_sample_mean_max_bp']:.2f}bp/month (sampling noise in the "
      f"default indicators, not bias)")

# Independent evaluation set: every risk number below is computed on this.
R_oos, _, _ = credit_scenarios(uni, n_scen=N_SCEN, seed=101, use_gpu=True)

GPU (CuPy): 0.099s  (excess returns)
default rate/bond 0.0358% (target 0.0388%, threshold calibrated on an independent draw)
4.5% of scenarios contain at least one default
mu vs realized scenario mean: max gap 4.17bp/month (sampling noise in the default indicators, not bias)


### What the tail is made of

Spread widening alone gives a fat but continuous left tail. **Defaults** give
the discrete jumps — and because they hit at the *issuer* level, holding
several bonds of one name turns one credit event into several simultaneous
losses. That is exactly the risk the per-issuer concentration limit exists to
control: the risk model and the limit sheet are describing the same thing.

In [5]:
losses = -(R_oos @ uni.w_benchmark)
cvar, var = empirical_cvar(losses, ALPHA)
q = np.percentile(losses, [50, 90, 95, 99, 99.9])
print(f"benchmark monthly excess-return loss quantiles:")
print(f"  median {q[0]:+.2%}   p90 {q[1]:+.2%}   p95 {q[2]:+.2%}   "
      f"p99 {q[3]:+.2%}   p99.9 {q[4]:+.2%}")
print(f"  VaR95 {var:.2%}    CVaR95 {cvar:.2%}")

benchmark monthly excess-return loss quantiles:
  median -0.20%   p90 +1.05%   p95 +1.49%   p99 +2.77%   p99.9 +6.11%
  VaR95 1.49%    CVaR95 2.42%


## 3. The limit sheet

Limits are derived from benchmark exposures plus explicit slack, so the index
is feasible by construction and the *slack* — how far the portfolio may stray —
is the tunable quantity, which is how mandates are written.

Two details the same review forced, both worth stating out loud:

- **DTS is bounded on both sides.** A one-sided cap lets the optimizer collapse
  credit beta toward zero — for a benchmarked mandate that is an enormous
  unstated tracking-error position, and it makes the cap itself inert. The
  floor is what forces a credit *selection* view instead of simple de-risking.
- **Rating bands are multiplicative for small buckets.** An additive ±10% band
  on a 4.7% single-B weight is not a band, it is permission to zero the bucket.
  `min(additive, ½ × benchmark weight)` gives small buckets a real floor, so a
  small allocation is a decision rather than a corner the parameterization
  handed over for free.

In [6]:
limits = default_limits(uni, turnover=0.25)
print(limits.describe())

model = build_mean_cvar(
    R, mu, alpha=ALPHA, risk_aversion=3.0, w_max=limits.w_max,
    constraint_builder=credit_constraint_builder(uni, limits),
)
print(f"\nLP: ~{model.n_assets + model.n_scenarios + 1:,} variables, "
      f"{model.nnz_estimate:,} nnz in the scenario block")
print("families:", ", ".join(model.extra_constraint_names))

issuer_cap=2.65%, w_max=3.23%, cell_dur_tol=±0.25y, total_dur=6.24±0.10y, DTS∈[829, 1,013]bp-y, turnover=25% one-way

LP: ~20,501 variables, 10,040,000 nnz in the scenario block
families: budget, w_max, cvar_tail, rating_bands_lo, rating_bands_hi, sector_caps, issuer_caps, cell_duration_match, total_duration_match, dts_cap, dts_floor, turnover


## 4. Solve on the GPU — and check whether the limits survived

A first-order solver returns a point that satisfies its *residual tolerance*.
That is not the same as satisfying a *desk limit*: cuOpt's fast 1e-4 solve
leaves a turnover-budget breach a risk manager would not sign off. Run
`python scripts/run_credit.py` for the full five-way comparison. Here we use a
configuration whose limits actually hold.

In [7]:
import cvxpy as cp

opts = dict(solver_method="PDLP", pdlp_solver_mode="Stable2",
            absolute_primal_tolerance=1e-4, relative_primal_tolerance=1e-4,
            absolute_dual_tolerance=1e-4, relative_dual_tolerance=1e-4,
            absolute_gap_tolerance=1e-4, relative_gap_tolerance=1e-4,
            time_limit=300.0, crossover=1)
model.problem.solve(solver=cp.CUOPT, verbose=False, warm_start=False, **opts)
w = model.weights()
print(model.problem.status, f"objective {model.problem.value:.6f}")

rep = exposure_report(uni, w, limits)
viol, which = worst_violation(rep)
print(f"worst relative limit violation: {viol:.2e}"
      + (f"  ({which})" if viol > 0 else "  — all limits respected"))
rep[rep["binding"]][["family", "item", "portfolio", "benchmark", "limit"]]

optimal objective -0.056245
worst relative limit violation: 5.86e-14  (total duration)


,family,item,portfolio,benchmark,limit
3,rating,B,0.023489,0.046978,"[2.3%, 14.7%]"
4,rating,BB,0.061899,0.123798,"[6.2%, 22.4%]"
5,rating,BBB,0.344629,0.444629,"[34.5%, 54.5%]"
14,issuer,max issuer weight,0.026473,0.017648,<= 2.65%
17,duration,7-12y contribution,2.061814,2.311814,±0.25y of bench
19,duration,total duration,6.135567,6.235567,±0.10y of bench
21,dts,DTS vs floor (bp-y),829.191806,921.324229,>= 829
22,turnover,one-way turnover,0.250000,0.000000,<= 25.0%


## 5. Does the answer make credit sense?

This matters more than any timing: a GPU that solves the wrong problem
quickly is worth nothing. Note the two CVaR columns — the optimizer fits the
worst 5% of the scenarios it was shown, so **only the out-of-sample column is
a risk reduction**; the in-sample one is a fit statistic.

In [8]:
rows = []
for label, ww in (("benchmark", uni.w_benchmark), ("incumbent", uni.w0),
                  ("CVaR-optimal", w)):
    c_is, _ = empirical_cvar(-(R @ ww), ALPHA)
    c_oos, v_oos = empirical_cvar(-(R_oos @ ww), ALPHA)
    st = portfolio_stats(uni, ww)
    rows.append({"portfolio": label, "exp.excess": float(mu @ ww),
                 "CVaR_in": c_is, "CVaR_oos": c_oos, "VaR_oos": v_oos,
                 "OAS(bp)": st["oas_bp"], "duration": st["duration"],
                 "DTS": st["dts_bp_years"], "positions": st["n_positions"],
                 "issuers": st["n_issuers"]})
comp = pd.DataFrame(rows).set_index("portfolio")
display(comp.round(4))

base, opt = comp.loc["benchmark"], comp.loc["CVaR-optimal"]
print(f"CVaR vs benchmark:  in-sample {1 - opt['CVaR_in']/base['CVaR_in']:+.1%}"
      f"   out-of-sample {1 - opt['CVaR_oos']/base['CVaR_oos']:+.1%}"
      f"   <- the second number is the real one")

,exp.excess,CVaR_in,CVaR_oos,VaR_oos,OAS(bp),duration,DTS,positions,issuers
portfolio,,,,,,,,,
benchmark,0.0012,0.0232,0.0242,0.0149,148.8655,6.2356,921.3242,500,200
incumbent,0.0012,0.0222,0.0232,0.0142,147.2401,6.0973,889.8646,325,176
CVaR-optimal,0.0012,0.0191,0.0205,0.0129,139.0161,6.1356,829.1918,270,155


CVaR vs benchmark:  in-sample +17.4%   out-of-sample +15.3%   <- the second number is the real one


In [9]:
tilt = pd.DataFrame({
    "optimal": uni.bonds.assign(x=w).groupby("rating", observed=True)["x"].sum(),
    "benchmark": uni.bonds.assign(x=uni.w_benchmark).groupby(
        "rating", observed=True)["x"].sum(),
})
tilt["active"] = tilt["optimal"] - tilt["benchmark"]
tilt.round(4)

,optimal,benchmark,active
rating,,,
A,0.3688,0.2979,0.0709
AA,0.1656,0.0676,0.0980
AAA,0.0356,0.0191,0.0165
B,0.0235,0.0470,-0.0235
BB,0.0619,0.1238,-0.0619
BBB,0.3446,0.4446,-0.1000


The optimizer moves up in quality — out of BBB and high yield, into A/AA/AAA —
and lands **on the floors**: the single-B, BB and BBB rating floors, the DTS
floor, the aggregate duration band and the turnover budget are all binding. It
would de-risk further if the mandate let it. That is a credit portfolio
manager's decision reproduced by an LP, with the limits doing exactly the work
limits are supposed to do.

Two honest caveats a credit PM will raise immediately, both deferred:
**no transaction costs** (high yield has the widest bid-ask, so a cost model
would damp the up-in-quality tilt), and **no rating migration** — historically
the dominant investment-grade credit-loss driver. Both are additive to this
framework rather than changes to it.

Next: **M3**, the scaling study (where does the GPU actually win?) and the
interactive re-solve demo built on cuOpt's warm start.